<a href="https://colab.research.google.com/github/Aniruddha-png/ECSR/blob/main/recognise_using_doodle.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import zipfile, os

zip_path = "/content/drive/MyDrive/Doodle Dataset/archive_Doodles.zip"  # path in Drive
extract_dir = "/content/doodle_dataset"
os.makedirs(extract_dir, exist_ok=True)

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_dir)

print("Files extracted to:", extract_dir)


Mounted at /content/drive
Files extracted to: /content/doodle_dataset


In [ ]:
import os

# List everything in current directory
print(os.listdir("/content"))


['.config', 'drive', 'doodle_dataset', 'sample_data']


In [ ]:
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, BatchNormalization
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

In [ ]:
# Step 1: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Step 2: Set the path to your dataset folder
# Replace 'doodle_dataset' with the exact folder name where your images are
data_path = '/content/drive/MyDrive/doodle_dataset/'


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!ls "/content/drive/MyDrive/"

'10202983_2024_020_copy (2)_page-0001.jpg'
 4701423b-7149-498a-8a6e-7378c3ed0908.jpg
 4th_send.xlsx
'4th Time Table files.xls'
'5. DSA - Trees.gdoc'
'7. DSA - Sorting.gslides'
 abf9f002-3d4b-423a-9a2e-5eb252518507.jpg
'Aniruddha Ghosh(2305919-B2) (BEE ASSIGNMENT -1).pdf'
 ANIRUDDHA-GHOSH_Resume.pdf
'Aniruddha Ghosh Roll No-2305919 Assignment-II.pdf'
'Aniruddha Ghosh  Roll No- 2305919.pdf'
 AntiraggingAffidavitForm.pdf
'CAMPUS RESELLING BY AMRIT.pdf'
'Ch-06 Relational Algebra (1).gslides'
'Ch-06 Relational Algebra.gslides'
'Ch-07 Relational Database Design.gslides'
'Ch12 Multithreading.gdoc'
'Ch5_Class&Object.gdoc'
 Classroom
'Colab Notebooks'
'DM Note (1).gdoc'
'Doodle Dataset'
'OB Assignment - 01_copy.pdf'
'ORGANISATIONAL CHANGE.gslides'
'ORGANISATIONAL STRUCTURE.gslides'
'Programming Lab Lessson Plan 2024.docx'
 Resume
'Screenshot 2025-07-27 152951.png'
 text_datasets
'Turing Machine(TM)_NEW.gslides'
'Voice Dataset'


In [ ]:
import zipfile
import os

zip_path = '/content/drive/MyDrive/Doodle_Dataset/archive_Doodles.zip'
extract_path = '/content/drive/MyDrive/Doodle Dataset/unzipped/'

os.makedirs(extract_path, exist_ok=True)

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Unzipped dataset at:", extract_path)
print("Subfolders (classes):", os.listdir(extract_path))


KeyboardInterrupt: 

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# =============================
# Step 3: Image Data Preprocessing
# =============================
import os
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Path to unzipped dataset folder (update if different)
data_path = '/content/drive/MyDrive/Doodle Dataset/unzipped/'

# Verify the path and list classes
if not os.path.exists(data_path):
    raise ValueError(f"Dataset path not found: {data_path}")
else:
    print(f"Dataset found at: {data_path}")
    print("Class subfolders:", os.listdir(data_path))

# Image parameters
img_height, img_width = 28, 28
batch_size = 64

# Data augmentation for training
train_datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2,      # 20% validation split
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.1,
    shear_range=0.1,
    horizontal_flip=True
)

# Training data generator
train_generator = train_datagen.flow_from_directory(
    data_path,
    target_size=(img_height, img_width),
    color_mode='grayscale',
    batch_size=batch_size,
    class_mode='sparse',
    subset='training',
    shuffle=True
)

# Validation data generator
val_generator = train_datagen.flow_from_directory(
    data_path,
    target_size=(img_height, img_width),
    color_mode='grayscale',
    batch_size=batch_size,
    class_mode='sparse',
    subset='validation',
    shuffle=False
)

Dataset found at: /content/drive/MyDrive/Doodle Dataset/
Class subfolders: ['archive_Doodles.zip']
Found 0 images belonging to 0 classes.
Found 0 images belonging to 0 classes.


In [ ]:

# =============================
# Step 4: Build CNN Model
# =============================
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, BatchNormalization

num_classes = len(train_generator.class_indices)

model = Sequential([
    Conv2D(32, (3,3), activation='relu', input_shape=(img_height,img_width,1)),
    BatchNormalization(),
    MaxPooling2D(2,2),

    Conv2D(64, (3,3), activation='relu'),
    BatchNormalization(),
    MaxPooling2D(2,2),

    Conv2D(128, (3,3), activation='relu'),
    BatchNormalization(),
    MaxPooling2D(2,2),

    Flatten(),
    Dense(256, activation='relu'),
    Dropout(0.5),
    Dense(num_classes, activation='softmax')
])

In [ ]:
# =============================
# Step 5: Compile Model
# =============================
from tensorflow.keras.optimizers import Adam

model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 26, 26, 32)     │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 26, 26, 32)     │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 13, 13, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 11, 11, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 11, 11, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 5, 5, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 3, 3, 128)      │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 3, 3, 128)      │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 1, 1, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │        33,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 0)              │             0 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 126,592 (494.50 KB)

 Trainable params: 126,144 (492.75 KB)

 Non-trainable params: 448 (1.75 KB)

In [ ]:
# =============================
# Step 6: Callbacks
# =============================
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3)
checkpoint = ModelCheckpoint('/content/drive/MyDrive/best_doodle_model.h5', monitor='val_accuracy', save_best_only=True)


In [ ]:
!ls "/content/drive/MyDrive/Doodle Dataset/"


archive_Doodles.zip


In [ ]:
# =============================
# Step 7: Train Model
# =============================
epochs = 20
history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=epochs,
    callbacks=[early_stop, reduce_lr, checkpoint]
)


/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


ValueError: The PyDataset has length 0

In [ ]:
# ==============================
# Training Loop with tqdm
# ==============================
def train_model(model, train_loader, val_loader, criterion, optimizer, device, num_epochs=5):
    model.to(device)

    for epoch in range(num_epochs):
        # ---------- Training ----------
        model.train()
        running_loss, correct, total = 0.0, 0, 0

        for images, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs} [Training]", leave=False):
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

        avg_train_loss = running_loss / len(train_loader)
        train_accuracy = 100 * correct / total

        # ---------- Validation ----------
        model.eval()
        val_loss, correct, total = 0.0, 0, 0
        with torch.no_grad():
            for images, labels in tqdm(val_loader, desc=f"Epoch {epoch+1}/{num_epochs} [Validation]", leave=False):
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                loss = criterion(outputs, labels)
                val_loss += loss.item()
                _, predicted = torch.max(outputs, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()

        avg_val_loss = val_loss / len(val_loader)
        val_accuracy = 100 * correct / total

        # ---------- Results ----------
        print(f"Epoch [{epoch+1}/{num_epochs}] "
              f"Train Loss: {avg_train_loss:.4f} | Train Acc: {train_accuracy:.2f}% | "
              f"Val Loss: {avg_val_loss:.4f} | Val Acc: {val_accuracy:.2f}%", flush=True)

    print("✅ Training complete!")

In [ ]:
!pip install tqdm

In [ ]:
# ==============================
# Run Training
# ==============================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
num_classes = len(train_loader.dataset.dataset.classes)  # for ImageFolder Subset

model = SimpleCNN(num_classes)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

train_model(model, train_loader, val_loader, criterion, optimizer, device, num_epochs=10)


KeyboardInterrupt: 

In [ ]:
plt.plot(history['train_acc'], label="Train Acc")
plt.plot(history['val_acc'], label="Val Acc")
plt.xlabel("Epochs")
plt.ylabel("Accuracy")
plt.legend()
plt.show()


In [ ]:
plt.plot(history['train_loss'], label="Train Loss")
plt.plot(history['val_loss'], label="Val Loss")
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.legend()
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

classes = ["Happy","Sad","Angry","Surprise","Neutral"]
acc = [0.91,0.87,0.83,0.89,0.92]  # replace with your computed per-class acc

plt.bar(classes, acc)
plt.xlabel("Emotion")
plt.ylabel("Accuracy")
plt.title("Per-class Accuracy")
plt.show()


In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

cm = confusion_matrix(y_true, y_pred)
ConfusionMatrixDisplay(cm, display_labels=classes).plot(cmap="Blues")
plt.show()


In [ ]:
from sklearn.metrics import roc_curve, auc
from sklearn.preprocessing import label_binarize
from sklearn.metrics import RocCurveDisplay

y_bin = label_binarize(y_true, classes=range(len(classes)))
for i in range(len(classes)):
    fpr, tpr, _ = roc_curve(y_bin[:, i], y_score[:, i])
    RocCurveDisplay(fpr=fpr, tpr=tpr, estimator_name=classes[i]).plot()
plt.show()


In [ ]:
from sklearn.metrics import precision_recall_curve, PrecisionRecallDisplay

for i in range(len(classes)):
    prec, rec, _ = precision_recall_curve(y_bin[:, i], y_score[:, i])
    PrecisionRecallDisplay(prec, rec).plot()
plt.title(classes[i])
plt.show()


In [ ]:
optimizers = ["Adam","SGD","RMSProp"]
acc = [0.92,0.88,0.90]
plt.bar(optimizers, acc)
plt.ylabel("Accuracy")
plt.title("Optimizer Comparison")
plt.show()


In [ ]:
acts = ["ReLU","Tanh","Sigmoid"]
acc = [0.91,0.87,0.82]
plt.bar(acts, acc)
plt.ylabel("Accuracy")
plt.title("Activation Comparison")
plt.show()


In [ ]:
plt.plot(history['train_acc'], label="Train Acc")
plt.plot(history['val_acc'], label="Val Acc")
plt.xlabel("Epochs")
plt.ylabel("Accuracy")
plt.legend()
plt.title("Epoch-wise Accuracy")
plt.show()


In [ ]:
batch_sizes = [1, 8, 16, 32]
latencies = [12, 5, 3, 2]  # ms
accuracies = [90, 89, 88, 87]  # %

plt.plot(latencies, accuracies, marker="o")
plt.xlabel("Latency (ms)")
plt.ylabel("Accuracy (%)")
plt.title("Latency vs Accuracy")
plt.show()


In [ ]:
wrong = np.where(y_pred != y_true)[0]
# plot first 10 misclassified samples
for i in wrong[:10]:
    plt.imshow(X[i].squeeze(), cmap="gray")
    plt.title(f"True: {classes[y_true[i]]}, Pred: {classes[y_pred[i]]}")
    plt.show()
